In [1]:
!pip install torch torchvision opencv-python pillow matplotlib ipywidgets

In [2]:
import sys
print(sys.version)

3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


In [ ]:
pip install notebook jupyterlab

In [ ]:
!pip install torch torchvision opencv-python pillow matplotlib ipywidgets


In [1]:
!pip install jupyterlab-widgets

In [ ]:
!pip install matplotlib

In [2]:
import sys
!{sys.executable} -m pip install -U ipython

   ---------------------------------------- 0.0/625.7 kB ? eta -:--:--
   ---------------- ----------------------- 262.1/625.7 kB ? eta -:--:--
   ---------------------------------------- 625.7/625.7 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: ipython
    Found existing installation: ipython 9.9.0
    Uninstalling ipython-9.9.0:
      Successfully uninstalled ipython-9.9.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spyder 6.0.7 requires ipython!=8.17.1,<9.0.0,>=8.13.0; python_version > "3.8", but you have ipython 9.12.0 which is incompatible.
spyder-kernels 3.0.5 requires ipython!=8.17.1,<9,>=8.13.0; python_version > "3.8", but you have ipython 9.12.0 which is incompatible.


In [3]:
conda install ipython=8.17.1

Jupyter detected...
3 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - defaults
Platform: win-64
Solving environment: failed

Note: you may need to restart the kernel to use updated packages.



PackagesNotFoundError: The following packages are not available from current channels:

  - ipython=8.17.1

Current channels:

  - defaults

To search for alternate channels that may provide the conda package you're
looking for, navigate to

    https://anaconda.org

and use the search bar at the top of the page.




In [4]:
import os

YOLO_ROOT = "datasets/bone fracture detection.v4-v4.yolov8"
OUT_ROOT  = "datasets/fracture_classification"
SPLITS = ["train", "valid"]


In [5]:
import io #for image upload
import numpy as np 
import torch 
import torch.nn as nn
import torchvision.transforms as T
from torchvision import models #image pre-processing
from PIL import Image 
import cv2 #heatmaps
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, clear_output

DEVICE ="cuda" if torch.cuda.is_available() else "cpu"
CLASS_NAMES = ["Anomaly", "Normal"]


In [6]:
#Loading a pre-trained neural network
def load_model():
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, 2)
    model.load_state_dict(
        torch.load("models/fracture_resnet18_best.pth", map_location=DEVICE)
    )
    model.eval().to(DEVICE)
    return model

model = load_model()


In [7]:
print(type(model))


<class 'torchvision.models.resnet.ResNet'>


In [8]:
#ensuring the neural network can interpret the image 
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

def preprocess(pil_img: Image.Image) -> torch.Tensor:
    return transform(pil_img.convert("RGB")).unsqueeze(0).to(DEVICE)

In [9]:

uploader = widgets.FileUpload(accept=".png, .jpg, .jpeg", multiple=False)
run_btn = widgets.Button(description="Run Analysis", button_style="success")
out = widgets.Output()

display(widgets.VBox([
    widgets.HTML("<h3>XAI X-RAY Prototype (Notebook)</h3>"),
    uploader, 
    run_btn, 
    out
]))

print("✓ Widgets created successfully!")

✓ Widgets created successfully!


In [16]:
import datetime
import io
from IPython.display import clear_output


class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, inp, out):
            self.activations = out.detach()

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, input_tensor, class_idx: int):
        self.model.zero_grad()
        logits = self.model(input_tensor)
        score = logits[:, class_idx].sum()
        score.backward(retain_graph=True)

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)

        cam = (weights * self.activations).sum(dim=1, keepdim=True)

        cam = torch.relu(cam)

        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam


def overlay_heatmap(pil_img, heatmap, alpha=0.45):
    """Overlay heatmap on original image"""
    img = np.array(pil_img.convert("RGB"))
    
    # Resize image to match heatmap dimensions
    img_resized = cv2.resize(img, (heatmap.shape[1], heatmap.shape[0]))

    heatmap_uint8 = np.uint8(255 * heatmap)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(img_resized, 1-alpha, heatmap_color, alpha, 0)

    return overlay


# Choosing target layer for Grad-CAM
target_layer = model.layer4[-1].conv2
cam_engine = GradCAM(model, target_layer)


def on_run_clicked(_):
    with out:
        clear_output()

        if not uploader.value:
            print("Please upload an X-ray image first.")
            return

        # --- Reading my uploaded image ---
        uploaded = uploader.value
        if isinstance(uploaded, dict):
            uploaded_file = next(iter(uploaded.values()))
            img_bytes = uploaded_file["content"]
        elif isinstance(uploaded, (list, tuple)):
            uploaded_file = uploaded[0]
            img_bytes = uploaded_file["content"]
        else:
            raise ValueError(f"Unexpected uploader.value type: {type(uploaded)}")

        pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

        # Preprocessing
        x = preprocess(pil_img)

        # Predict
        with torch.no_grad():
            logits = model(x)
            probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

        pred_idx = int(probs.argmax())
        conf = float(probs[pred_idx])

        
        # Generate Grad-CAM heatmap
        heatmap = cam_engine.generate(x, pred_idx)
        
        #Resizing heatmap for smooth visualization
        heatmap_resized = cv2.resize(heatmap, (224, 224))
        
        # Creating overlay with resized heatmap
        overlay = overlay_heatmap(pil_img, heatmap_resized)

        # Print prediction results
        print(f"Prediction: {CLASS_NAMES[pred_idx]}")
        print(f"Confidence: {conf:.1%}")
        print("="*50)

        
        plt.figure(figsize=(15, 5))
        
        # Original X-Ray
        plt.subplot(1, 3, 1)
        plt.title("Original X-Ray", fontsize=14, fontweight='bold')
        plt.imshow(pil_img, cmap='gray')
        plt.axis("off")
        
        # Grad-CAM Heatmap
        plt.subplot(1, 3, 2)
        plt.title("AI Attention Heatmap\n(Red = High Focus)", fontsize=14, fontweight='bold')
        plt.imshow(heatmap_resized, cmap='jet', interpolation='bilinear')  # ✅ Smooth display
        plt.axis("off")
        
        # Overlay
        plt.subplot(1, 3, 3)
        color = 'red' if 'Anomaly' in CLASS_NAMES[pred_idx] else 'green'
        plt.title(f"Prediction: {CLASS_NAMES[pred_idx]}\nConfidence: {conf:.1%}", 
                 fontsize=14, fontweight='bold', color=color)
        plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        
        plt.tight_layout()
        plt.show()

        
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        original_filename = uploaded_file.get('name', 'image.png')
        out_path = f"results/gradcam_{original_filename}_{timestamp}.png"
        
        os.makedirs("results", exist_ok=True)
        overlay_rgb = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
        Image.fromarray(overlay_rgb).save(out_path)
        print(f"\n✓ Saved overlay as: {out_path}")


run_btn.on_click(on_run_clicked)

print("✓ GradCAM system ready!")
print("Upload an X-ray image and click 'Run Analysis' to see results.")

✓ GradCAM system ready!
Upload an X-ray image and click 'Run Analysis' to see results.


In [23]:
uploader = widgets.FileUpload (accept= ".png, .jpg, .jpeg", multiple=False)
run_btn =widgets.Button(description ="Run Analysis" , button_style="success")
out = widgets.Output()

display(widgets.VBox([
    widgets.HTML("<h3>XAI X-RAY Prototype (Notebook)</h3>"),
    uploader, 
    run_btn, 
    out
]))

In [24]:
import os

for split in ["train", "valid", "test"]:
    img_path = os.path.join(YOLO_ROOT, split, "images")
    lbl_path = os.path.join(YOLO_ROOT, split, "labels")
    print(split)
    print("  images:", img_path, "->", os.path.exists(img_path))
    print("  labels:", lbl_path, "->", os.path.exists(lbl_path))


train
  images: datasets/bone fracture detection.v4-v4.yolov8\train\images -> False
  labels: datasets/bone fracture detection.v4-v4.yolov8\train\labels -> False
valid
  images: datasets/bone fracture detection.v4-v4.yolov8\valid\images -> True
  labels: datasets/bone fracture detection.v4-v4.yolov8\valid\labels -> True
test
  images: datasets/bone fracture detection.v4-v4.yolov8\test\images -> False
  labels: datasets/bone fracture detection.v4-v4.yolov8\test\labels -> False


In [25]:
import os

print("YOLO_ROOT:", YOLO_ROOT)
print("Subfolders inside YOLO_ROOT:")
print([f for f in os.listdir(YOLO_ROOT) if os.path.isdir(os.path.join(YOLO_ROOT, f))])


YOLO_ROOT: datasets/bone fracture detection.v4-v4.yolov8
Subfolders inside YOLO_ROOT:
['.ipynb_checkpoints', 'valid']


In [26]:
# configuring the files
YOLO_ROOT = "datasets/bone fracture detection.v4-v4.yolov8"
OUT_ROOT  = "datasets/fracture_classification"

print("YOLO_ROOT set to:", YOLO_ROOT)


YOLO_ROOT set to: datasets/bone fracture detection.v4-v4.yolov8


In [27]:
import os, shutil, random

random.seed(42)

# Source YOLO valid split
img_dir = os.path.join(YOLO_ROOT, "valid", "images")
lbl_dir = os.path.join(YOLO_ROOT, "valid", "labels")

# Output folders
train_anom = os.path.join(OUT_ROOT, "train", "Anomaly")
train_norm = os.path.join(OUT_ROOT, "train", "Normal")
val_anom   = os.path.join(OUT_ROOT, "valid", "Anomaly")
val_norm   = os.path.join(OUT_ROOT, "valid", "Normal")

for p in [train_anom, train_norm, val_anom, val_norm]:
    os.makedirs(p, exist_ok=True)

def is_anomaly_label(label_path):
    # anomaly only if YOLO label exists AND is not empty
    return os.path.exists(label_path) and os.path.getsize(label_path) > 0

# Collecting all images
imgs = [f for f in os.listdir(img_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))]

print("Total images found:", len(imgs))

random.shuffle(imgs)

#splitting
cut = int(0.8 * len(imgs))
train_imgs = imgs[:cut]
val_imgs   = imgs[cut:]

def copy_split(img_list, out_anom, out_norm):
    counts = {"Anomaly": 0, "Normal": 0}
    for img_name in img_list:
        stem = os.path.splitext(img_name)[0]
        lbl_path = os.path.join(lbl_dir, stem + ".txt")
        src_img = os.path.join(img_dir, img_name)

        if is_anomaly_label(lbl_path):
            shutil.copy2(src_img, os.path.join(out_anom, img_name))
            counts["Anomaly"] += 1
        else:
            shutil.copy2(src_img, os.path.join(out_norm, img_name))
            counts["Normal"] += 1
    return counts

train_counts = copy_split(train_imgs, train_anom, train_norm)
val_counts   = copy_split(val_imgs,   val_anom,   val_norm)

print("\n Classification dataset created")
print("Train counts:", train_counts)
print("Val counts:", val_counts)
print("\nSaved to:", OUT_ROOT)


Total images found: 348

 Classification dataset created
Train counts: {'Anomaly': 138, 'Normal': 140}
Val counts: {'Anomaly': 35, 'Normal': 35}

Saved to: datasets/fracture_classification
